# PyMMM Pipeline: ND2 → Zarr

This notebook demonstrates the full PyMMM pipeline:
1. Load an ND2 file lazily
2. Register (drift-correct) the data
3. Detect feeding lanes
4. Detect trenches within lanes
5. Extract trenches to a compressed zarr store

Each stage checkpoints to a companion zarr store, so you can restart the kernel and resume from any checkpoint.

In [1]:
## enable jupyter autoreload magics
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import sys
import os
sys.path.insert(0, "..")

In [3]:
from pymmm import ND2Experiment, Registrator, LaneDetector, TrenchDetector, Extractor
from pymmm.checkpoint import CompanionStore
from dask.distributed import Client, LocalCluster
import hvplot.xarray
import hvplot

hvplot.extension('bokeh')

# Start a dask cluster for parallelisation
cluster = LocalCluster(processes=True, threads_per_worker=1, n_workers=os.cpu_count())
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 32
Total threads: 32,Total memory: 61.91 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38227,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:36957,Total threads: 1
Dashboard: http://127.0.0.1:40359/status,Memory: 1.93 GiB
Nanny: tcp://127.0.0.1:34545,


2026-03-02 17:35:08,887 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='localhost:8787', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/home/georgeos/GitHub/PyMMM/.pixi/envs/default/lib/python3.14/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/home/georgeos/GitHub/PyMMM/.pixi/envs/default/lib/python3.14/site-packages/tornado/web.py", line 3388, in wrapper
    return method(self, *args, **kwargs)
  File "/home/georgeos/GitHub/PyMMM/.pixi/envs/default/lib/python3.14/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expired. Configure the app with a larger value 

## 1. Load ND2 file

In [4]:
exp = ND2Experiment("/data/scientific_data/20260109_SB7_exit_snake.nd2") 
exp.select_times(0,750)

# Optional: crop to a spatial ROI
exp.select_roi(y=(570, 1000), x=(0, -1))

print(exp)

# Interactive browse raw data
exp.data.hvplot.image(
    x="X", y="Y", cmap="Greys_r", dynamic=True,
    rasterize=True, widget_location="top", aspect="equal"
)

ND2Experiment: 20260109_SB7_exit_snake
  Path: /data/scientific_data/20260109_SB7_exit_snake.nd2
  Dims: T=750 × P=36 × C=2 × Y=430 × X=1305
  FOVs: 36  (XYPos:0, XYPos:1, XYPos:2...)
  Channels: PC, mCherry
  Timepoints: 750
  Pixel size: 0.1079 µm
  Time interval: 30000.5 ms


BokehModel(combine_events=True, render_bundle={'docs_json': {'672fc5bd-6e82-4097-963e-3622c3991b41': {'version…

In [5]:
# Optional: discard unwanted FOVs
# exp.discard_fovs(["xy029", "xy030"])

## 2. Companion store

In [6]:
store = CompanionStore.for_experiment(exp)
store

CompanionStore(/data/scientific_data/20260109_SB7_exit_snake.pymmm.zarr, [registration, lane_detection, trench_detection])

## 3. Registration (Checkpoint 1)

Register the experiment to correct for stage drift.

In [7]:
#store.reset("registration")
if store.has_registration():
    print("Loading registration from checkpoint...")
    reg = Registrator.load(exp, store)
else:
    reg = Registrator(
        exp, store,
        registration_channel="PC",  # ← Change to your phase-contrast channel
        mode="first",
        rotation=0,               # ← Adjust rotation if needed
        roi={"y": (0, -1), "x": (0, -1)},
    )
    reg.compute_mean_images(plot=False)
    reg.compute_tmats(plot=True)  # n_jobs=-1 by default (all cores)
    reg.save()

Loading registration from checkpoint...


In [8]:
# Interactive browse registered data
reg.get_stabilized_data().hvplot.image(
    x="X", y="Y", cmap="Greys_r", dynamic=True,
    rasterize=True, widget_location="top", aspect="equal"
)

BokehModel(combine_events=True, render_bundle={'docs_json': {'e8b82011-3da2-4659-901f-55129d34f851': {'version…

In [9]:
# Drift diagnostics for a specific FOV
#reg.plot_drift(fov=2)

## 4. Lane Detection (Checkpoint 2)

Detect the feeding lane y-positions in each FOV.

In [10]:
#store.reset("lane_detection")
if store.has_lane_detection():
    print("Loading lane detection from checkpoint...")
    lane_det = LaneDetector.load(exp, reg, store)
else:
    lane_det = LaneDetector(exp, reg, store, detection_channel="PC")

Loading lane detection from checkpoint...


In [11]:
# Tune sliders, browse FOVs, then click "Apply to all FOVs"
#lane_det.interactive_detect_lanes()

In [12]:
#for fov, lanes in lane_det._lanes.items():
#    for lane in lanes:
#        lane.orientation = 1  

# Then re-save
#lane_det.save()

In [13]:
# After tuning sliders and clicking "Apply to all FOVs", save the results
#lane_det.save()

In [14]:
# Inspect another FOV
# lane_det.plot_fov(fov=1)

## 5. Trench Detection (Checkpoint 3)

Detect trench x-positions within each lane.

In [15]:
#store.reset("trench_detection")
if store.has_trench_detection():
    print("Loading trench detection from checkpoint...")
    trench_det = TrenchDetector.load(exp, reg, lane_det, store)
else:
    trench_det = TrenchDetector(exp, reg, lane_det, store, detection_channel="PC")

Loading trench detection from checkpoint...


In [16]:
# Tune sliders, browse FOVs, then click "Apply to all FOVs"
#trench_det.interactive_detect_trenches()

In [17]:
# After tuning sliders and clicking "Apply to all FOVs", save the results
#trench_det.save()

In [18]:
# Trench table
trench_det.get_trench_table()

,trench_id,fov,lane_index,x_left,x_right,y_top,y_bottom,orientation,needs_flip
0,0,XYPos:0,0,42,76,60,224,1,False
1,1,XYPos:0,0,86,120,60,224,1,False
2,2,XYPos:0,0,133,167,60,224,1,False
3,3,XYPos:0,0,178,212,60,224,1,False
4,4,XYPos:0,0,224,258,60,224,1,False
...,...,...,...,...,...,...,...,...,...
1004,1004,XYPos:35,0,1087,1121,63,227,1,False
1005,1005,XYPos:35,0,1132,1166,63,227,1,False
1006,1006,XYPos:35,0,1177,1211,63,227,1,False
1007,1007,XYPos:35,0,1222,1256,63,227,1,False


In [19]:
# Preview a single trench before full extraction
#from pymmm.plotting import plot_trench_preview
#preview = trench_det_extractor_preview = Extractor(exp, reg, trench_det)
#single = preview.extract_single_trench(trench_id=0)
#plot_trench_preview(single, trench_id=0, n_frames=8)

## 6. Extraction

Extract all trenches to a compressed zarr store.

In [20]:
extractor = Extractor(exp, reg, trench_det)
print(extractor)  # dimensions, chunks, size estimate
extractor.preview()

Extractor: 20260109_SB7_exit_snake
  Output: /data/scientific_data/20260109_SB7_exit_snake.trenches.zarr
  Shape: Trench=1009 × T=750 × C=2 × Y=164 × X=34
  Chunks: (1, 750, 1, 164, 34)
  Dtype: uint16
  Size (uncompressed): 15.7 GB
  Trenches: 1009 across 36 FOVs
  Trench size: 164 × 34 px (17.7 × 3.7 µm)
  Channels: PC, mCherry
  Timepoints: 750


<xarray.DataArray 'concatenate-cd0d4334da5d509a770f33e3b2707a43' (Trench: 1009,
                                                                  T: 750, C: 2,
                                                                  Y: 164, X: 34)> Size: 17GB
dask.array<concatenate, shape=(1009, 750, 2, 164, 34), dtype=uint16, chunksize=(1, 750, 1, 164, 34), chunktype=numpy.ndarray>
Coordinates:
  * Trench   (Trench) int64 8kB 0 1 2 3 4 5 6 ... 1003 1004 1005 1006 1007 1008
  * C        (C) <U7 56B 'PC' 'mCherry'
  * Y        (Y) int64 1kB 0 1 2 3 4 5 6 7 8 ... 156 157 158 159 160 161 162 163
  * X        (X) int64 272B 0 1 2 3 4 5 6 7 8 9 ... 25 26 27 28 29 30 31 32 33
Dimensions without coordinates: T

In [21]:
extractor.extract(compressor='zstd', clevel=9, show_progress=True)

Extracting FOVs:   0%|          | 0/36 [00:00<?, ?it/s]

Extraction complete: /data/scientific_data/20260109_SB7_exit_snake.trenches.zarr
  Shape: (1009, 750, 2, 164, 34), Chunks: (1, 750, 1, 164, 34)


## 7. Verify output

In [ ]:
import zarr

z = zarr.open(str(extractor.output_path), mode='r')
attrs = dict(z.attrs)
meta = attrs["source_acquisition_metadata"]
subset = attrs["source_subset_metadata"]

print("Shape:", z['data'].shape)
print("Chunks:", z['data'].chunks)
print("Attr keys:", sorted(attrs))
print("Metadata version:", attrs["source_metadata_version"])
print("Methods subset:", {
    "date": meta["structured"]["text_info"].get("date"),
    "optics": meta["structured"]["text_info"].get("optics"),
    "channels": attrs["channel_names"],
    "pixel_size_um": meta["file_summary"]["pixel_size_um"],
    "time_interval_ms": meta["file_summary"]["time_interval_ms"],
})
print("Subset metadata:", subset)